# Connections Solver — walkthrough
Live tour of the pipeline for the TA check-in. Each section runs the real code in `src/`.
Run from the repo root folder (the first cell handles the path).

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import pandas as pd
from src import data
from src.data import read_raw, build_puzzles, split, pretty, RAW_DIR
from src.puzzle import LEVELS

## Step 1 — The shared data shape (`src/puzzle.py`)
Every module speaks the same two types, so Kevin (data, scorers, search) and Emme (game, policy, LLM, eval)
can work in parallel. Both are **frozen dataclasses**: immutable (no accidental edits mid-experiment) and
hashable (usable as cache keys — the scorers cache embeddings per puzzle).

In [ ]:
from src.puzzle import Puzzle, Group, LEVELS
g = Group(words=("BASS", "PIKE", "SOLE", "TROUT"), name="FISH", level=0)
print(g)
print("level 0 =", LEVELS[g.level])

In [ ]:
try:
    g.level = 3
except Exception as e:
    print("frozen ->", type(e).__name__)
cache = {g.words: "cached embeddings would live here"}   # hashable -> works as a dict key
cache[g.words]

## Step 2 — Data: load, clean, split
**Source:** `eric27n/NYT-Connections` on Hugging Face, **one row per word** (16 rows per game).

In [ ]:
raw = read_raw()          # downloads once into data/raw/, then cached
print(raw.shape)
raw.head(16)              # one full game

### The NULL trap
By default pandas converts the *text* `NULL`, `NA`, `NAN`, `NONE` into missing values — but those are real
Connections answers. Compare default loading vs. ours (everything read as plain text, auto-NaN off):

In [ ]:
csv = sorted(RAW_DIR.glob("*.csv"))[0]
naive = pd.read_csv(csv)
lost = naive[naive["Word"].isna()].index
print(f"default pandas: {len(lost)} words became NaN")
pd.DataFrame({"default pandas": naive.loc[lost, "Word"], "our loader": raw.loc[lost, "word"],
              "game": raw.loc[lost, "game_id"], "group": raw.loc[lost, "group_name"]})

### Group rows → `Puzzle` objects, drop invalid ones
Keep a game only if: 16 unique non-empty words, 4 groups of 4, levels exactly {0,1,2,3}.

In [ ]:
puzzles = build_puzzles(raw)   # prints kept / dropped counts and reasons

### Chronological split
Test = newest 100 (least likely in LLM training data), val = 50 before that, train = rest.
Never train on the future and test on the past; settings are tuned on **val only**.

In [ ]:
train, val, test = split(puzzles)
pd.DataFrame([{"split": n, "puzzles": len(ps), "from": ps[0].date, "to": ps[-1].date}
              for n, ps in [("train", train), ("val", val), ("test", test)]])

### Sanity check by eye
One puzzle as a board — check it against the NYT Connections archive for that date.

In [ ]:
print(pretty(test[0]))

## Step 3 — Embedding scorers (`src/baseline.py`)
**Score of a 4-word group = average cosine similarity over its 6 word pairs.** Higher = more likely a real group.

Two ways to get a word's vector (the professor's question: single words vs. context?):
- **st_single** — sentence-transformer (MiniLM) on each word *alone*. BASS gets one vector no matter the puzzle.
- **contextual** — the *same model*, but all 16 words go in as one input `"bass, pike, drum, ..."`.
  Self-attention mixes information between words, so BASS's vector depends on its neighbors.
  A word split into several subword tokens gets the mean of those tokens' vectors.

Same model in both → any difference is caused by context alone.

In [ ]:
from src.data import load_split
from src.baseline import make_scorer
val = load_split("val")
scorers = {k: make_scorer(k) for k in ["st_single", "contextual"]}
p = val[0]
print(pretty(p))

### Similarity heatmaps
Words sorted by true group, so a perfect scorer shows four bright 4×4 blocks on the diagonal.

In [ ]:
import matplotlib.pyplot as plt
order = [w for g in p.groups for w in g.words]
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (k, sc) in zip(axes, scorers.items()):
    S = sc.similarity_matrix(p.words)
    idx = [p.words.index(w) for w in order]
    im = ax.imshow(S[idx][:, idx], cmap="viridis")
    ax.set_xticks(range(16), order, rotation=90); ax.set_yticks(range(16), order)
    for b in range(1, 4): ax.axhline(4*b - .5, c="w", lw=1); ax.axvline(4*b - .5, c="w", lw=1)
    ax.set_title(k); fig.colorbar(im, ax=ax, fraction=.046)
plt.tight_layout()

### True groups vs. the highest-scoring groups
If a wrong group outscores a true one, that's a red herring fooling the scorer.

In [ ]:
import itertools
truth = {frozenset(g.words): LEVELS[g.level] for g in p.groups}
for k, sc in scorers.items():
    print(f"--- {k}")
    for g in p.groups:
        print(f"  TRUE {LEVELS[g.level]:<6} {sc.score(p.words, g.words):.3f}  {g.words}")
    top = sorted(itertools.combinations(p.words, 4), key=lambda g: -sc.score(p.words, g))[:5]
    for g in top:
        tag = truth.get(frozenset(g), "wrong")
        print(f"  top5 {tag:<6} {sc.score(p.words, g):.3f}  {g}")

### Across all val puzzles
`true` / `random` = mean score of true groups vs. random 4-word groups (true should be clearly higher).
Per-color columns = mean true-group score (purple should be lowest). `top1=true` = how often the single
best-scoring group of all 1,820 is a real group — a preview of how greedy search will do.

In [ ]:
from src.baseline import sanity
from src.puzzle import LEVELS
rows = []
for k, sc in scorers.items():
    r = sanity(sc, val)
    rows.append({"scorer": k, "true": r["mean_true"], "random": r["mean_random"],
                 **dict(zip(LEVELS, r["mean_true_by_level"])), "top1=true": r["top1_is_true_group"]})
pd.DataFrame(rows).round(3)